<a href="https://colab.research.google.com/github/apache/beam/blob/master/examples/notebooks/get-started/iceberg-cdc-quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ###### Licensed to the Apache Software Foundation (ASF), Version 2.0 (the "License")

# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements. See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership. The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License. You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied. See the License for the
# specific language governing permissions and limitations
# under the License.

# Apache Iceberg CDC quickstart

In this notebook, you'll keep one Iceberg table in sync with another using change data capture
(CDC), with a streaming pipeline that applies each change as it lands. You'll write everything in
[Beam SQL](https://beam.apache.org/documentation/dsls/sql/overview/), following the
[Iceberg CDC quickstart](https://beam.apache.org/documentation/io/built-in/iceberg-cdc/).

The quickstart uses three Beam SQL shells. Here, each shell is a session, named on the first line
of a cell:

* `live` shows the changes to an `orders` table as they arrive.
* `mirror` runs a streaming pipeline that applies those changes to a second table,
  `orders_mirror`.
* `writer` changes `orders` and queries both tables.

Streaming statements keep running in the background, so you can go on to the next cell while they
run. Run the cells in order, one at a time.

## Set up

Pick where the pipelines run:

* **Locally**: everything runs in this notebook's runtime, on the DirectRunner, and the tables live
  in a Hadoop catalog on its disk. There's nothing else to set up.
* **Dataflow**: the tables live in a
  [Lakehouse](https://beam.apache.org/documentation/io/built-in/iceberg/#choose-your-catalog)
  catalog in your Google Cloud project, and the mirror pipeline runs on Dataflow. You need a
  project with billing enabled, and you'll be asked to sign in.

The first run builds Beam SQL, which takes a few minutes. Running the cell again starts over, with
empty tables.

In [ ]:
#@title Set up Beam SQL
#@markdown Pick where the pipelines run, then run this cell.
run_on = "Locally"  #@param ["Locally", "Dataflow"]
#@markdown **For Dataflow only:** your Google Cloud project, and a bucket for the tables. Leave the bucket empty to use one named after the project. The cell creates the bucket, and a Lakehouse catalog on it, if they don't exist yet.
project = ""  #@param {type:"string"}
bucket = ""  #@param {type:"string"}
region = "us-central1"  #@param {type:"string"}

# The rest of this cell runs Beam SQL from the notebook. Each session is a JDBC connection to
# Beam SQL, like a shell of its own. Streaming statements run in the background, and what they
# show keeps updating under the cell that started them.

import glob
import html
import http.server
import importlib
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid

from IPython import get_ipython
from IPython.display import HTML, JSON, display

try:
  from google.colab import output as _colab_output
except ImportError:
  _colab_output = None

BEAM_VERSION = "2.79.0"
LOCAL_WAREHOUSE = "file:///tmp/beam-iceberg-cdc-quickstart"

_KINDS = {"INSERT": "+I", "UPDATE_BEFORE": "-U", "UPDATE_AFTER": "+U", "DELETE": "-D"}
_GREEN, _RED, _AMBER, _GRAY = "#1e8e3e", "#d93025", "#e37400", "#80868b"
_STATE_COLORS = {"STARTING": _AMBER, "RUNNING": _GREEN, "STOPPING": _AMBER, "FAILED": _RED}
_FONT = "ui-monospace, SFMono-Regular, Menlo, Consolas, 'Liberation Mono', monospace"
_NUMBER = re.compile(r"-?\d+(\.\d+)?")
_MAX_ROWS = 500

if "_stop_all" in globals():
  _stop_all()  # Stop what an earlier run of this cell started.

_config = {"setup": None, "dataflow": None}
_sessions = {}
_live = {}
_current = threading.local()


class BeamSqlError(Exception):
  """An error that the notebook shows as just its message, in red."""

  def _render_traceback_(self):
    return ["\x1b[31m%s\x1b[0m" % self]


# Setup


def _setup(run_on, project="", bucket="", region=""):
  """Builds Beam SQL, starts it, and connects to the catalog. Returns the runner to use."""
  cloud = run_on == "Dataflow"
  if cloud:
    project = project.strip()
    if not project:
      raise BeamSqlError(
          "To run on Dataflow, fill in your Google Cloud project, and run this cell again.")
    bucket = bucket.strip().removeprefix("gs://").strip("/")
    bucket = bucket or re.sub(r"[^a-z0-9-]+", "-", project.lower()) + "-iceberg-cdc"
    region = region.strip() or "us-central1"
  _stop_all()
  _config["setup"] = None
  classpath = os.environ.get("BEAM_SQL_CLASSPATH")
  version = None
  if not classpath:
    version = _beam_version()
    classpath = os.path.expanduser(
        "~/.beam/cache/beam-%s_runner-%s_ios-iceberg.jar"
        % (version, "dataflow" if cloud else "direct"))
  started_with = globals().get("_jvm_classpath")
  if started_with and started_with != classpath:
    raise BeamSqlError(
        "Beam SQL is already running with other settings. To switch, restart the session "
        "(Runtime > Restart session), then run this cell again.")
  if cloud:
    with _Step("Signing in to Google Cloud") as step:
      _sign_in(project)
      step.done("Signed in to Google Cloud")
    _set_up_google_cloud(project, bucket, region)
  java_home = _java()
  if version:
    _build(version, java_home, cloud, classpath)
  _start(java_home, classpath)
  catalog = _catalog_sql(cloud, project, bucket)
  with _Step("Connecting to the catalog") as step:
    _config["dataflow"] = (
        {"project": project, "region": region, "bucket": bucket} if cloud else None)
    try:
      dropped = _fresh_start(catalog)
    except BeamSqlError as e:
      hint = ""
      if cloud and re.search(r"credential|401|403|unauthenticated", str(e), re.IGNORECASE):
        hint = ("\nIf Beam SQL can't use your Google credentials, run "
                "!gcloud auth application-default login in a new cell, then this cell again.")
      raise BeamSqlError("Couldn't connect to the catalog: %s%s" % (e, hint)) from None
    _config["setup"] = [catalog, "USE DATABASE quickstart.shop"]
    step.done(
        "Connected to the catalog"
        + (", and dropped %s from an earlier run" % " and ".join(dropped) if dropped else ""))
  display(HTML(
      '<div style="margin: 10px 0 4px">Every session below starts by connecting to the catalog:'
      '</div><pre style="margin: 0; padding: 8px 12px; border-radius: 8px; background: rgba(128,'
      ' 128, 128, .12); font-size: 13px">%s</pre>'
      % html.escape(catalog + ";\nUSE DATABASE quickstart.shop;")))
  return "DataflowRunner" if cloud else "DirectRunner"


def _beam_version():
  """BEAM_VERSION if it's released, and its nightly snapshot until then."""
  if BEAM_VERSION.endswith("-SNAPSHOT"):
    return BEAM_VERSION
  url = (
      "https://repo1.maven.org/maven2/org/apache/beam/beam-sdks-java-core/{0}/"
      "beam-sdks-java-core-{0}.pom".format(BEAM_VERSION))
  try:
    urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30).close()
    return BEAM_VERSION
  except urllib.error.HTTPError as e:
    if e.code == 404:
      return BEAM_VERSION + "-SNAPSHOT"
    raise


def _sign_in(project):
  os.environ["GOOGLE_CLOUD_PROJECT"] = project
  if _colab_output is not None:
    from google.colab import auth
    auth.authenticate_user(project_id=project)


def _set_up_google_cloud(project, bucket, region):
  """Creates the bucket and the catalog if they're missing, and gives Dataflow access."""
  if not shutil.which("gcloud"):
    raise BeamSqlError("Setting up Google Cloud needs the gcloud CLI. Install it, and try again.")
  with _Step("Turning on the Lakehouse and Dataflow APIs") as step:
    _gcloud("services", "enable", "biglake.googleapis.com", "dataflow.googleapis.com",
            "--project", project)
    step.done("The Lakehouse and Dataflow APIs are on")
  with _Step("Looking for bucket gs://%s" % bucket) as step:
    if _gcloud("storage", "buckets", "describe", "gs://" + bucket, check=False).returncode:
      step.update("Creating bucket gs://%s" % bucket)
      _gcloud("storage", "buckets", "create", "gs://" + bucket, "--project", project,
              "--location", region, "--uniform-bucket-level-access")
      step.done("Created bucket gs://%s" % bucket)
    else:
      step.done("Using bucket gs://%s" % bucket)
  catalog = ["biglake", "iceberg", "catalogs"]
  with _Step("Looking for Lakehouse catalog %s" % bucket) as step:
    described = _gcloud(*catalog, "describe", bucket, "--project", project, "--format=json",
                        check=False)
    if described.returncode:
      step.update("Creating Lakehouse catalog %s" % bucket)
      _gcloud(*catalog, "create", bucket, "--project", project, "--catalog-type=gcs-bucket",
              "--credential-mode=vended-credentials")
      described = _gcloud(*catalog, "describe", bucket, "--project", project, "--format=json")
      step.done("Created Lakehouse catalog %s" % bucket)
    else:
      step.done("Using Lakehouse catalog %s" % bucket)
  with _Step("Letting the catalog and Dataflow's workers use the bucket") as step:
    problems = []

    def grant(*args):
      result = _gcloud(*args, check=False)
      if result.returncode:
        problems.append(result.stderr.strip().splitlines()[-1])

    # The catalog hands out short-lived credentials for the bucket, made with its own account.
    catalog_account = json.loads(described.stdout).get("biglake-service-account")
    if catalog_account:
      grant("storage", "buckets", "add-iam-policy-binding", "gs://" + bucket,
            "--member=serviceAccount:" + catalog_account, "--role=roles/storage.objectUser")
    else:
      problems.append("Couldn't find the catalog's service account, to let it use the bucket.")
    # Dataflow's workers run as the project's Compute Engine default service account.
    number = _gcloud("projects", "describe", project, "--format=value(projectNumber)").stdout
    worker = "serviceAccount:%s-compute@developer.gserviceaccount.com" % number.strip()
    for role in ("roles/dataflow.worker", "roles/biglake.editor",
                 "roles/serviceusage.serviceUsageConsumer"):
      grant("projects", "add-iam-policy-binding", project, "--member=" + worker,
            "--role=" + role, "--condition=None")
    grant("storage", "buckets", "add-iam-policy-binding", "gs://" + bucket,
          "--member=" + worker, "--role=roles/storage.objectAdmin")
    if problems:
      step.warn(
          "Couldn't grant every role. If the pipelines can't reach the catalog, ask a project "
          "owner to run this cell.\n" + "\n".join(problems))
    else:
      step.done("The catalog and Dataflow's workers can use the bucket")


def _gcloud(*args, check=True):
  result = subprocess.run(["gcloud", *args, "--quiet"], capture_output=True, text=True)
  if check and result.returncode:
    raise BeamSqlError("gcloud %s failed:\n%s" % (" ".join(args[:4]), result.stderr.strip()))
  return result


def _java():
  """Returns the home of a JDK that runs Beam SQL, Java 17 to 23, installing one if needed."""
  with _Step("Looking for Java") as step:
    javas = [os.path.join(os.environ.get("JAVA_HOME", "/nonexistent"), "bin", "java"),
             shutil.which("java") or "/nonexistent"]
    javas += sorted(glob.glob("/usr/lib/jvm/*/bin/java"), reverse=True)
    homes = [os.path.dirname(os.path.dirname(os.path.realpath(j)))
             for j in javas if os.path.exists(j)]
    for home in dict.fromkeys(homes):
      version = _java_version(home)
      # Java 24 and later can't run the Hadoop catalog.
      if version and 17 <= version <= 23 and os.path.exists(_libjvm(home)):
        step.done("Java %d" % version)
        return home
    if os.geteuid() != 0 or not shutil.which("apt-get"):
      raise BeamSqlError(
          "Beam SQL needs Java 17 to 23. Install it, point JAVA_HOME at it, and try again.")
    step.update("Installing Java 21")
    env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
    for args in (["update"], ["install", "-y", "openjdk-21-jdk-headless"]):
      result = subprocess.run(["apt-get", "-qq", *args], capture_output=True, text=True, env=env)
      if result.returncode:
        raise BeamSqlError("Couldn't install Java 21:\n" + _tail(result.stdout + result.stderr))
    for home in glob.glob("/usr/lib/jvm/java-21-openjdk-*"):
      if os.path.exists(_libjvm(home)):
        step.done("Installed Java 21")
        return home
    raise BeamSqlError("Couldn't find Java 21 after installing it.")


def _java_version(home):
  try:
    result = subprocess.run([os.path.join(home, "bin", "java"), "-version"],
                            capture_output=True, text=True, timeout=60)
  except (OSError, subprocess.SubprocessError):
    return None
  match = re.search(r'version "(\d+)(?:\.(\d+))?', result.stderr)
  if not match:
    return None
  major = int(match.group(1))
  return int(match.group(2) or 0) if major == 1 else major


def _libjvm(home):
  return os.path.join(home, "lib", "server", "libjvm.so")


def _build(version, java_home, cloud, jar):
  """Builds the Beam SQL shell's JAR with beam-sql.sh, unless it's built already."""
  name = "Beam SQL %s with Iceberg%s" % (version, " and the Dataflow runner" if cloud else "")
  if os.path.exists(jar):
    _Step.show("✓", name, _GREEN)
    return
  note = " Beam %s isn't out yet, so this uses its nightly build." % BEAM_VERSION
  with _Step("Building %s. The first time takes a few minutes.%s"
             % (name, note if version.endswith("-SNAPSHOT") else "")) as step:
    started = time.time()
    script = os.path.expanduser("~/.beam/beam-sql.sh")
    os.makedirs(os.path.dirname(script), exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/apache/beam/master/scripts/beam-sql.sh", script)
    os.chmod(script, 0o755)
    args = [script, "--version", version, "--io", "iceberg"]
    if cloud:
      args += ["--runner", "dataflow"]
    env = dict(os.environ, JAVA_HOME=java_home,
               PATH=os.path.join(java_home, "bin") + os.pathsep + os.environ["PATH"])
    try:
      # The script starts the shell once it's built, and "!quit" closes it.
      result = subprocess.run(args, input="!quit\n", capture_output=True, text=True, env=env,
                              cwd=os.path.expanduser("~"), timeout=1800)
      output = result.stdout + result.stderr
    except subprocess.TimeoutExpired as e:
      partial = e.output.decode(errors="replace") if isinstance(e.output, bytes) else e.output
      output = "Timed out after 30 minutes.\n%s" % (partial or "")
    if not os.path.exists(jar):
      raise BeamSqlError("Couldn't build Beam SQL:\n" + _tail(output))
    step.done("Built %s in %s" % (name, _clock(time.time() - started, words=True)))


def _start(java_home, classpath):
  """Starts the JVM with Beam SQL, if it isn't running yet, and hooks it up to the notebook."""
  global jpype
  with _Step("Starting Beam SQL") as step:
    try:
      import jpype
    except ImportError:
      subprocess.run([sys.executable, "-m", "pip", "install", "-q", "JPype1"], check=True)
      importlib.invalidate_caches()
      import jpype
    if not jpype.isJVMStarted():
      jpype.startJVM(
          _libjvm(java_home),
          # Keep the libraries' logs out of the notebook. Errors still show up, under the
          # statements that fail.
          "-Dslf4j.provider=org.slf4j.helpers.NOP_FallbackServiceProvider",
          "-Dslf4j.internal.verbosity=ERROR",
          "-Dorg.apache.commons.logging.Log=org.apache.commons.logging.impl.NoOpLog",
          classpath=classpath.split(os.pathsep),
          convertStrings=True)
      globals()["_jvm_classpath"] = classpath
      logging = jpype.JPackage("java").util.logging
      logging.Logger.getLogger("").setLevel(logging.Level.OFF)
    globals()["_viewer"] = _make_viewer()
    jpype.JClass("org.apache.beam.sdk.extensions.sql.impl.rel.LiveResult").setViewer(_viewer)
    if _colab_output is not None:
      _colab_output.register_callback("beam_sql.render", _render_callback)
      _serve()
    step.done("Started Beam SQL")


def _catalog_sql(cloud, project, bucket):
  if cloud:
    properties = [
        ("type", "rest"),
        ("uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog"),
        ("warehouse", "gs://" + bucket),
        ("header.x-goog-user-project", project),
        ("rest.auth.type", "google"),
        ("io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO"),
        ("header.X-Iceberg-Access-Delegation", "vended-credentials"),
    ]
  else:
    properties = [("type", "hadoop"), ("warehouse", LOCAL_WAREHOUSE)]
  return "CREATE CATALOG quickstart TYPE 'iceberg'\nPROPERTIES (\n%s\n)" % ",\n".join(
      "  '%s' = '%s'" % p for p in properties)


def _fresh_start(catalog):
  """Creates the database, and drops the tables an earlier run left. Returns the dropped ones."""
  session = _Session("setup", [catalog])
  try:
    session.run("CREATE DATABASE IF NOT EXISTS quickstart.shop")
    session.run("USE DATABASE quickstart.shop")
    tables = [row[0] for row in session.run("SHOW TABLES").rows]
    dropped = [table for table in ("orders", "orders_mirror") if table in tables]
    for table in dropped:
      session.run("DROP TABLE " + table)
    return dropped
  finally:
    session.close()


class _Step:
  """A line of setup progress, which gets a ✓ when the step is done."""

  def __init__(self, text):
    self.text, self.final, self.icon, self.color = text, None, "✓", _GREEN
    self.handle = display(HTML(self._html("…", text, _GRAY)), display_id=True)

  def __enter__(self):
    return self

  def update(self, text):
    self.text = text
    self.handle.update(HTML(self._html("…", text, _GRAY)))

  def done(self, text):
    self.final = text

  def warn(self, text):
    self.final, self.icon, self.color = text, "⚠", _AMBER

  def __exit__(self, kind, error, traceback):
    if error is None:
      self.handle.update(HTML(self._html(self.icon, self.final or self.text, self.color)))
    else:
      self.handle.update(HTML(self._html("✗", self.text, _RED)))

  @staticmethod
  def show(icon, text, color):
    display(HTML(_Step._html(icon, text, color)))

  @staticmethod
  def _html(icon, text, color):
    return ('<div style="font-family: %s; font-size: 13px; white-space: pre-wrap; margin: 2px 0">'
            '<span style="color: %s">%s</span> %s</div>' % (_FONT, color, icon, html.escape(text)))


# Sessions


def _session(name):
  """Returns the session called `name`, and opens it the first time."""
  if not _config["setup"]:
    raise BeamSqlError("Run the setup cell at the top first.")
  if name not in _sessions:
    _sessions[name] = _Session(name, _config["setup"])
  return _sessions[name]


def stop_all():
  """Stops every streaming statement, and closes the sessions."""
  running = _stop_all()
  if not running:
    display(HTML(_note("Nothing was running.")))
  for item in running:
    _show_live(item)
  dataflow = _config["dataflow"]
  if dataflow:
    display(HTML(_note(
        "The tables, the bucket gs://{0} and the Lakehouse catalog {0} stay in your project, so "
        "you can come back to them. Delete them when you no longer need them."
        .format(dataflow["bucket"]))))


def _stop_all():
  running = [item for item in list(_live.values()) if not item.done]
  for item in running:
    item.stop()
  deadline = time.time() + 20
  for item in running:
    item.wait(max(0, deadline - time.time()))
  for session in list(_sessions.values()):
    session.close()
  _sessions.clear()
  return running


class _Session:
  """A JDBC connection to Beam SQL. Like a shell, it runs one statement at a time."""

  def __init__(self, name, setup):
    self.name = name
    self.runner = "DirectRunner"
    self.running = None
    self.connection = jpype.JClass("java.sql.DriverManager").getConnection("jdbc:beam:")
    for sql in setup:
      self.run(sql)

  def use_runner(self, runner):
    """Runs this session's pipelines on `runner`. Returns the SET statements that did it."""
    if runner not in ("DirectRunner", "DataflowRunner"):
      raise BeamSqlError("The runner can be DirectRunner or DataflowRunner, not %s." % runner)
    if runner == self.runner:
      return []
    statements = ["SET runner = DirectRunner"]
    if runner == "DataflowRunner":
      dataflow = _config["dataflow"]
      if not dataflow:
        raise BeamSqlError("To run on Dataflow, pick Dataflow in the setup cell, and run it again.")
      statements = [
          "SET runner = DataflowRunner",
          "SET project = '%s'" % dataflow["project"],
          "SET region = '%s'" % dataflow["region"],
          "SET tempLocation = 'gs://%s/tmp'" % dataflow["bucket"],
          "SET maxNumWorkers = 1",
      ]
    done = [self.run(sql) for sql in statements]
    self.runner = runner
    return done

  def run(self, sql):
    """Runs one statement, and returns what to show for it."""
    if self.running is not None and not self.running.done:
      self.running.stop()
      self.running.wait(60)
    if _is_streaming(sql):
      kind = _Stream if re.match(r"\s*SELECT\b", sql, re.IGNORECASE) else _Background
      self.running = kind(self, sql)
      return self.running
    statement = self.connection.createStatement()
    try:
      if statement.execute(sql):
        return _Table(sql, statement.getResultSet())
      return _Done(sql, statement.getUpdateCount())
    except jpype.JException as e:
      raise BeamSqlError(_message(e)) from None
    finally:
      statement.close()

  def close(self):
    try:
      self.connection.close()
    except Exception:  # pylint: disable=broad-except
      pass


def _split_statements(sql):
  """Splits `sql` at semicolons outside quotes, and drops -- comments."""
  statements, current, quote, i = [], [], None, 0
  while i < len(sql):
    c = sql[i]
    if quote:
      current.append(c)
      if c == quote:
        quote = None
    elif c in "'\"`":
      quote = c
      current.append(c)
    elif sql.startswith("--", i):
      end = sql.find("\n", i)
      i = len(sql) if end < 0 else end
      continue
    elif c == ";":
      statements.append("".join(current).strip())
      current = []
    else:
      current.append(c)
    i += 1
  statements.append("".join(current).strip())
  return [s for s in statements if s]


def _is_streaming(sql):
  return re.search(r"'streaming'\s*=\s*true", sql, re.IGNORECASE) is not None


# What statements show


class _Done:
  """A statement that finished without rows to show."""

  _SUMMARY = re.compile(
      r"(CREATE (?:EXTERNAL )?TABLE(?: IF NOT EXISTS)?|CREATE (?:DATABASE|CATALOG)"
      r"(?: IF NOT EXISTS)?|DROP (?:TABLE|DATABASE|CATALOG)(?: IF EXISTS)?|INSERT INTO"
      r"|ALTER TABLE|USE(?: DATABASE| CATALOG)?) ([\w.`\"]+)", re.IGNORECASE)

  def __init__(self, sql, count=-1):
    flat = " ".join(sql.split())
    match = self._SUMMARY.match(flat)
    if match:
      self.text = "%s %s" % (match.group(1).upper(), match.group(2))
    else:
      self.text = flat if len(flat) <= 80 else flat[:79] + "…"
    if count > 0 and flat.upper().startswith("INSERT"):
      self.text += " · %d %s" % (count, "row" if count == 1 else "rows")

  def html(self):
    return ('<div style="font-family: %s; font-size: 13px; opacity: .7; margin: 2px 0">✓ %s</div>'
            % (_FONT, html.escape(self.text)))


class _Table:
  """The rows a query returned."""

  def __init__(self, sql, results):
    match = re.search(r"\bFROM\s+([\w.]+)", sql, re.IGNORECASE)
    self.title = match.group(1) if match else " ".join(sql.split()).rstrip(";")
    meta = results.getMetaData()
    self.columns = [str(meta.getColumnLabel(i + 1)) for i in range(meta.getColumnCount())]
    self.rows = []
    while results.next():
      self.rows.append([_text(results.getObject(i + 1)) for i in range(len(self.columns))])

  def html(self):
    count = "%d %s" % (len(self.rows), "row" if len(self.rows) == 1 else "rows")
    return _frame(self.title, html.escape(count), _table(self.columns, [(None, r) for r in self.rows]))


class _Running:
  """A streaming statement, which runs on a thread of its own until it's stopped."""

  first_state = "STARTING"

  def __init__(self, session, sql):
    self.id = uuid.uuid4().hex
    self.session = session
    self.sql = sql
    self.state = self.first_state
    self.started = time.time()
    self.ended = None
    self.error = None
    self.stop_requested = False
    self.statement = session.connection.createStatement()
    _live[self.id] = self
    self.thread = threading.Thread(target=self._run, daemon=True)
    self.thread.start()

  @property
  def done(self):
    return self.ended is not None

  def stop(self):
    self.stop_requested = True
    if not self.done:
      self.state = "STOPPING"
      try:
        self.statement.cancel()
      except Exception:  # pylint: disable=broad-except
        pass

  def wait(self, seconds):
    self.thread.join(seconds)

  def _run(self):
    try:
      self.execute()
      self.state = "STOPPED" if self.stop_requested else "FINISHED"
    except jpype.JException as e:
      if self.stop_requested:
        self.state = "STOPPED"
      else:
        self.state, self.error = "FAILED", _message(e)
    except Exception as e:  # pylint: disable=broad-except
      self.state, self.error = "FAILED", str(e)
    finally:
      if self.state == "STOPPING":  # stop() came in as the statement ended.
        self.state = "STOPPED"
      self.ended = time.time()
      try:
        self.statement.close()
      except Exception:  # pylint: disable=broad-except
        pass

  def clock(self):
    return _clock((self.ended or time.time()) - self.started)

  def version(self):
    return "%s:%s" % (self.state, bool(self.error))


class _Stream(_Running):
  """A streaming query. The viewer adds its rows, with the kind of change each one is."""

  def __init__(self, session, sql):
    match = re.search(r"CHANGES\s*\(\s*([\w.]+)\s*\)", sql, re.IGNORECASE)
    self.title = "Changes to %s" % match.group(1) if match else "Live results"
    self.columns = []
    self.rows = []
    self.received = 0
    self.lock = threading.Lock()
    super().__init__(session, sql)

  def execute(self):
    _current.stream = self
    self.statement.executeQuery(self.sql).close()

  def add(self, kind, values):
    with self.lock:
      self.rows.append((kind, values))
      self.received += 1
      del self.rows[:-_MAX_ROWS]

  def version(self):
    return "%s:%d" % (super().version(), self.received)

  def status_html(self):
    rows = "%d %s" % (self.received, "row" if self.received == 1 else "rows")
    return _status_html(self.state, self.clock(), rows)

  def html(self):
    with self.lock:
      rows = list(self.rows)
    body = ""
    if self.columns:
      waiting = "Waiting for changes…" if not rows and not self.done else None
      body = _table(["op"] + self.columns, rows, waiting, kinds=True)
    return _frame(self.title, self.status_html(), body, error=self.error, scroll=True)


class _Background(_Running):
  """A streaming statement with no rows to show, like an INSERT that keeps a table in sync."""

  first_state = "RUNNING"

  def __init__(self, session, sql):
    target = re.search(r"INSERT\s+INTO\s+([\w.]+)", sql, re.IGNORECASE)
    source = re.search(r"CHANGES\s*\(\s*([\w.]+)\s*\)", sql, re.IGNORECASE)
    if target and source:
      self.title = "Keeping %s in sync with %s" % (target.group(1), source.group(1))
    else:
      self.title = "Streaming statement"
    self.dataflow = _config["dataflow"] if session.runner == "DataflowRunner" else None
    super().__init__(session, sql)

  def execute(self):
    self.statement.execute(self.sql)

  def status_html(self):
    return _status_html(self.state, "on Dataflow" if self.dataflow else "locally", self.clock())

  def html(self):
    note = None
    if self.dataflow:
      note = ('<a href="https://console.cloud.google.com/dataflow/jobs?project=%s" target="_blank"'
              ' style="color: inherit">See the job in the Dataflow console ↗</a>'
              % urllib.parse.quote(self.dataflow["project"]))
    return _frame(self.title, self.status_html(), note=note, error=self.error)


def _make_viewer():
  """The live view for streaming queries: it adds their rows to the stream that ran them."""
  time_unit = jpype.JClass("java.util.concurrent.TimeUnit")

  @jpype.JImplements("org.apache.beam.sdk.extensions.sql.impl.rel.LiveResult$Viewer")
  class NotebookViewer:

    @jpype.JOverride
    def show(self, result):
      stream = getattr(_current, "stream", None)
      if stream is None:
        return
      stream.columns = [str(name) for name in result.getSchema().getFieldNames()]
      stream.state = "RUNNING"
      while not stream.stop_requested and not result.isCancelRequested():
        change = result.poll(200, time_unit.MILLISECONDS)
        while change is not None:
          row = change.getRow()
          values = [_text(row.getValue(i)) for i in range(row.getFieldCount())]
          stream.add(_KINDS.get(str(change.getKind().name()), "?"), values)
          change = result.poll(0, time_unit.MILLISECONDS)
        if result.isFinished():
          break

  return NotebookViewer()


# Showing results


def _sql_magic(line, cell):
  """%%sql SESSION [--runner DirectRunner|DataflowRunner] [--watch SESSION]

  Runs the cell's statements in SESSION, a connection to Beam SQL of its own. With --watch, also
  shows the live view of another session's streaming query under the cell.
  """
  args = line.split()
  name = args.pop(0) if args and not args[0].startswith("--") else "default"
  options = dict(zip(args[::2], args[1::2]))
  if len(args) % 2 or set(options) - {"--runner", "--watch"}:
    raise BeamSqlError("Usage: %%sql SESSION [--runner RUNNER] [--watch SESSION]")
  session = _session(name)
  items = []
  try:
    if "--runner" in options:
      items += session.use_runner(options["--runner"])
    for sql in _split_statements(cell):
      items.append(session.run(sql))
  finally:
    _show(items)
  if "--watch" in options:
    watched = _sessions.get(options["--watch"])
    if watched and isinstance(watched.running, _Stream) and not watched.running.done:
      _show_live(watched.running)
    else:
      display(HTML(_note("Nothing is streaming in session %s." % options["--watch"])))


def _show(items):
  """Shows what a cell's statements returned. Tables in a row sit side by side."""
  tables = []
  for item in items + [None]:
    if isinstance(item, _Table):
      tables.append(item)
      continue
    if tables:
      display(HTML('<div style="display: flex; flex-wrap: wrap; gap: 16px; align-items: flex-start">'
                   '%s</div>' % "".join(t.html() for t in tables)))
      tables = []
    if isinstance(item, _Running):
      _show_live(item)
    elif item is not None:
      display(HTML(item.html()))


def _show_live(item):
  """Shows `item`, and keeps it up to date until it stops."""
  if _colab_output is not None:
    server = globals().get("_server")
    display(HTML(
        _POLL.replace("__BOX__", "beam-sql-" + uuid.uuid4().hex)
        .replace("__ID__", item.id)
        .replace("__VERSION__", item.version())
        .replace("__PORT__", str(server.server_address[1]) if server else "null")
        .replace("__HTML__", item.html())))
    return
  handle = display(HTML(item.html()), display_id=True)

  def refresh():
    while not item.done:
      time.sleep(1)
      handle.update(HTML(item.html()))

  threading.Thread(target=refresh, daemon=True).start()


# In Colab, the live view asks the kernel for updates. It asks a small web server in the kernel
# first, which answers even while a cell runs, and calls back into the kernel if it can't reach it.
_POLL = """
<div id="__BOX__">__HTML__</div>
<script>
(async () => {
  const box = document.getElementById("__BOX__");
  const port = __PORT__;
  let version = "__VERSION__";
  let viaServer = port !== null;
  let failures = 0;
  for (;;) {
    await new Promise(resolve => setTimeout(resolve, 500));
    let data;
    try {
      if (viaServer) {
        const response = await fetch(`https://localhost:${port}/live/__ID__`,
                                     {cache: "no-store", signal: AbortSignal.timeout(5000)});
        if (!response.ok) throw new Error(`HTTP ${response.status}`);
        data = await response.json();
      } else {
        const reply = await google.colab.kernel.invokeFunction("beam_sql.render", ["__ID__"], {});
        data = reply.data["application/json"];
      }
      failures = 0;
    } catch (e) {
      if (viaServer) {
        viaServer = false;
      } else if (++failures >= 5) {
        return;
      }
      continue;
    }
    if (data.html !== null && data.version !== version) {
      // Keep the rows scrolled where they were.
      const rows = box.querySelector(".bsql-body");
      const scrolled = rows ? rows.scrollTop : 0;
      box.innerHTML = data.html;
      const newRows = box.querySelector(".bsql-body");
      if (newRows) newRows.scrollTop = scrolled;
      version = data.version;
    } else if (data.status !== null) {
      const status = box.querySelector(".bsql-status");
      if (status) status.innerHTML = data.status;
    }
    if (data.done) return;
  }
})();
</script>
"""


def _payload(item):
  if item is None:
    return {"html": None, "status": None, "version": None, "done": True}
  return {"html": item.html(), "status": item.status_html(), "version": item.version(),
          "done": item.done}


def _render_callback(item_id):
  return JSON(_payload(_live.get(item_id)))


class _Handler(http.server.BaseHTTPRequestHandler):

  def do_GET(self):  # pylint: disable=invalid-name
    body = json.dumps(_payload(_live.get(self.path.rsplit("/", 1)[-1]))).encode()
    self.send_response(200)
    self.send_header("Content-Type", "application/json")
    self.send_header("Cache-Control", "no-store")
    # Don't save the answers in the notebook.
    self.send_header("x-colab-notebook-cache-control", "no-cache")
    self.send_header("Access-Control-Allow-Origin", "*")
    self.end_headers()
    self.wfile.write(body)

  def log_message(self, *args):
    pass


def _serve():
  """Starts the web server that the live view asks for updates, unless it's running already."""
  if globals().get("_server") is not None:
    return

  class Server(http.server.ThreadingHTTPServer):
    address_family = socket.AF_INET6
    daemon_threads = True

  try:
    server = Server(("::", 0), _Handler)
  except OSError:
    return  # The live view calls back into the kernel instead.
  threading.Thread(target=server.serve_forever, daemon=True).start()
  globals()["_server"] = server


def _status_html(state, *parts):
  return '<span style="color: %s">●</span> %s' % (
      _STATE_COLORS.get(state, _GRAY), html.escape(" · ".join((state,) + parts)))


def _frame(title, status, body="", error=None, note=None, scroll=False):
  out = [
      '<div style="font-family: %s; font-size: 13px; display: inline-block; min-width: 380px;'
      ' max-width: 100%%; border: 1px solid rgba(128, 128, 128, .35); border-radius: 8px;'
      ' overflow: hidden; margin: 2px 0">' % _FONT,
      '<div style="padding: 6px 12px; background: rgba(128, 128, 128, .12); display: flex;'
      ' justify-content: space-between; align-items: baseline; gap: 32px"><b>%s</b>'
      '<span class="bsql-status" style="white-space: nowrap">%s</span></div>'
      % (html.escape(title), status),
  ]
  if body:
    style = ("max-height: 320px; overflow: auto; display: flex; flex-direction: column-reverse"
             if scroll else "")
    out.append('<div class="bsql-body" style="%s"><div>%s</div></div>' % (style, body))
  if note:
    out.append('<div style="padding: 6px 12px">%s</div>' % note)
  if error:
    out.append('<div style="padding: 6px 12px; color: %s; white-space: pre-wrap">%s</div>'
               % (_RED, html.escape(error)))
  out.append("</div>")
  return "".join(out)


def _table(columns, rows, placeholder=None, kinds=False):
  """An HTML table of `rows`, which are (kind, values) pairs. With `kinds`, columns[0] is op."""
  cell = "padding: 3px 12px; white-space: nowrap"
  # Line numbers up on the right, headers included.
  numeric = [bool(rows) and all(_NUMBER.fullmatch(values[i]) for _, values in rows)
             for i in range(len(columns) - kinds)]
  numeric = [False] * kinds + numeric
  head = "".join(
      '<th style="%s; text-align: %s; border-bottom: 1px solid rgba(128, 128, 128, .35)">%s</th>'
      % (cell, "right" if numeric[i] else "left", html.escape(c)) for i, c in enumerate(columns))
  lines = []
  for kind, values in rows:
    cells = []
    if kind is not None:
      cells.append('<td style="%s; color: %s">%s</td>'
                   % (cell, _RED if kind.startswith("-") else _GREEN, kind))
    for value in values:
      align = "right" if _NUMBER.fullmatch(value) else "left"
      cells.append('<td style="%s; text-align: %s">%s</td>' % (cell, align, html.escape(value)))
    lines.append("<tr>%s</tr>" % "".join(cells))
  if placeholder and not rows:
    lines.append('<tr><td colspan="%d" style="%s; opacity: .6">%s</td></tr>'
                 % (len(columns), cell, html.escape(placeholder)))
  return '<table style="border-collapse: collapse; width: 100%%"><tr>%s</tr>%s</table>' % (
      head, "".join(lines))


def _note(text):
  return ('<div style="font-family: %s; font-size: 13px; opacity: .7; margin: 2px 0">%s</div>'
          % (_FONT, html.escape(text)))


def _text(value):
  return "null" if value is None else str(value)


def _clock(seconds, words=False):
  seconds = int(seconds)
  if words:
    return "%dm %02ds" % (seconds // 60, seconds % 60) if seconds >= 60 else "%ds" % seconds
  return "%02d:%02d" % (seconds // 60, seconds % 60)


def _tail(text, lines=40):
  return "\n".join(text.strip().splitlines()[-lines:])


def _message(error):
  """The innermost cause's message: the exceptions around it only wrap it."""
  cause = error
  while cause.getCause() is not None and cause.getCause() != cause:
    cause = cause.getCause()
  message = cause.getMessage()
  return str(message) if message else str(cause.getClass().getSimpleName())


if get_ipython() is not None:
  get_ipython().register_magic_function(_sql_magic, magic_kind="cell", magic_name="sql")

runner = _setup(run_on, project, bucket, region)

## 1. Watch a table's changes

Create the source table, `orders`, and open a live view of its changes. It's empty for now, but it
fills up when you write to the table in step 3. The view keeps running, so go on to the next
step.

In [ ]:
%%sql live
CREATE EXTERNAL TABLE orders (
  id BIGINT PRIMARY KEY NOT ENFORCED,
  item VARCHAR,
  qty INTEGER
)
TYPE 'iceberg';

SELECT * FROM CHANGES(orders)
WITH ('streaming' = true, 'starting_strategy' = 'earliest', 'poll_interval_seconds' = 1);

## 2. Keep a table in sync (streaming)

In a second session, you'll run a streaming pipeline that continuously observes `orders` and
applies its changes to a downstream table. First, create that table, `orders_mirror`:

In [ ]:
%%sql mirror
CREATE EXTERNAL TABLE orders_mirror (
  id BIGINT PRIMARY KEY NOT ENFORCED,
  item VARCHAR,
  qty INTEGER
)
TYPE 'iceberg';

Now continuously stream changes from `orders` into
`orders_mirror`. On the read side, select two Iceberg metadata columns by name:

- the change type (`_change_type`)
- the snapshot sequence number (`_commit_snapshot_sequence_number`)

On the write side, point `change_type_column` at the type. You don't need to set
`sequence_number_column`, because it already defaults to `_commit_snapshot_sequence_number`. The
sink drops both columns before it writes.

Streaming writes need a commit window, which you set with `triggering_frequency_seconds`. Streaming
reads need a polling interval, which you set with `poll_interval_seconds` (defaults to 60s). A
streaming read starts at the latest snapshot by default, so set `starting_strategy` to `earliest`
to also apply the changes made before the pipeline started.

`--runner $runner` runs the pipeline where you picked in the setup. On Dataflow, the cell shows the
`SET` statements that send it there, and the job takes a few minutes to start.

In [ ]:
%%sql mirror --runner $runner
INSERT INTO orders_mirror
WITH ('mode' = 'merge-on-read',
      'change_type_column' = '_change_type',
      'triggering_frequency_seconds' = 1)
SELECT *, _change_type, _commit_snapshot_sequence_number
FROM CHANGES(orders)
WITH ('streaming' = true,
      'starting_strategy' = 'earliest',
      'poll_interval_seconds' = 1);

## 3. Write some changes and watch them arrive

In a third session, check that both tables are there:

In [ ]:
%%sql writer
SHOW TABLES;

Now change `orders` in two rounds, one snapshot each. Each
change record carries two metadata values:

- the record's change type (`op`)
- the record's sequence number (`seq`), which goes up with each change to the same key. Both halves
  of an update (`UPDATE_BEFORE` and `UPDATE_AFTER`) share one.

`--watch live` shows the live view from step 1 again, under the cell, so you can watch the changes
arrive.

In [ ]:
%%sql writer --watch live
-- Snapshot 1: three new orders
INSERT INTO orders
WITH (
  'mode' = 'merge-on-read',
  'change_type_column' = 'op',
  'sequence_number_column' = 'seq')
  (id, item, qty, op, seq)
VALUES (1, 'apple', 3, 'INSERT', 1),
       (2, 'pear', 5, 'INSERT', 1),
       (3, 'plum', 2, 'INSERT', 1);

Soon you'll see three inserts show up in the live view, in any
order. Query both tables. Each should hold the same three orders. On Dataflow, the mirror catches up
once the job is running, so if `orders_mirror` is still empty, run the cell again in a minute.

In [ ]:
%%sql writer
SELECT * FROM orders;
SELECT * FROM orders_mirror;

Now make a second round of changes:

In [ ]:
%%sql writer --watch live
-- Snapshot 2: change order 1, delete order 2, add order 4
INSERT INTO orders
WITH (
  'mode' = 'merge-on-read',
  'change_type_column' = 'op',
  'sequence_number_column' = 'seq')
  (id, item, qty, op, seq)
VALUES (1, 'apple', 3, 'UPDATE_BEFORE', 2),
       (1, 'apple', 4, 'UPDATE_AFTER', 2),
       (2, 'pear', 5, 'DELETE', 2),
       (4, 'kiwi', 1, 'INSERT', 2);

You'll see four new changes stream into the live view: `+I` for
kiwi, `-D` for pear, and `-U` and `+U` for apple's old and new quantities. Query both tables again
to verify the changes were applied correctly:

In [ ]:
%%sql writer
SELECT * FROM orders;
SELECT * FROM orders_mirror;

`orders_mirror` should now match `orders`, in any order:

| id | item  | qty |
|---:|-------|----:|
| 1  | apple | 4   |
| 3  | plum  | 2   |
| 4  | kiwi  | 1   |

## Clean up

When you're done, stop the live view and the streaming pipeline. On Dataflow, this cancels the job.
A streaming Dataflow job keeps running, and costs money, until it's cancelled, even after you close
this notebook.

In [ ]:
stop_all()

## Learn more

* [Iceberg CDC quickstart](https://beam.apache.org/documentation/io/built-in/iceberg-cdc/), which also covers Java, Python and YAML
* [Stream changes between Iceberg tables with Apache Beam](https://beam.apache.org/blog/iceberg-cdc-sql/)
* [Beam SQL shell](https://beam.apache.org/documentation/dsls/sql/shell/), to run these
  statements on your own machine
* [Apache Iceberg I/O connector](https://beam.apache.org/documentation/io/built-in/iceberg/)